# 0. Install Seqeval

In [1]:
!git clone https://github.com/chakki-works/seqeval.git /tmp/seqeval_repo
!cp -r /tmp/seqeval_repo/seqeval /usr/local/lib/python3.13/dist-packages/
!rm -rf /tmp/seqeval_repo
!echo "Install Seq Eval completed"

Cloning into '/tmp/seqeval_repo'...
remote: Enumerating objects: 808, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 808 (delta 6), reused 5 (delta 5), pack-reused 798 (from 2)
Receiving objects: 100% (808/808), 185.39 KiB | 12.36 MiB/s, done.
Resolving deltas: 100% (461/461), done.
Install Seq Eval completed


# 1. Clone repository

In [2]:
!git clone -q https://github.com/ducbao210/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora.git
%cd Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora

/kaggle/working/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora


In [3]:
!pwd

/kaggle/working/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora


# 2. Dependencies

In [4]:
!pip install -q -r requirements-kaggle.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 34.1 MB/s eta 0:00:00


In [5]:
# Kaggle may ship an old optional torchao that newer PEFT rejects during LoRA setup.
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [6]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

def secret(name: str) -> str:
    try:
        return user_secrets.get_secret(name)
    except Exception as exc:
        print(f"Secret {name} is not available: {exc}")
        return ""

hf_token = secret("HF_TOKEN_NER")
checkpoint_repo = secret("CHECKPOINT_REPO")

with open('.env', 'w') as f:
    f.write(f"HF_TOKEN={hf_token}\n")
    f.write(f"HF_CHECKPOINT_REPO_ID={checkpoint_repo}\n")

print("Successfully created .env!")

Successfully created .env!


# 3. Download dataset

In [7]:
!python download_data.py

Downloaded: dev_word.conll, test_word.conll, train_word.conll


# 4. Training models

In [8]:
!python main.py \
  --mode train \
  --model all \
  --upload_checkpoint

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
config.json: 100%|█████████████████████████████| 557/557 [00:00<00:00, 1.91MB/s]
vocab.txt: 100%|█████████████████████████████| 895k/895k [00:00<00:00, 85.6MB/s]
bpe.codes: 100%|████████████████████████████| 1.14M/1.14M [00:00<00:00, 151MB/s]
tokenizer.json: 100%|███████████████████████| 3.13M/3.13M [00:00<00:00, 189MB/s]
2026-10-19 15:12:19 - model - INFO - Instantiating LSTM model (CRF=False)

 lstm_crf_False Parameters
  Total parameters:       19,776,532
  Trainable parameters:   19,776,532
  Frozen parameters:               0
  Trainable ratio:           100.00%

Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ..

In [9]:
# knowledge distillation: Teacher - Phobert; Student - BiLSTM
!python main.py --mode distill --model bilstm --upload_checkpoint

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 26846.78it/s]
2026-10-19 16:32:27 - model - INFO - vinai/phobert-base encoder: 14,175,744/134,998,272 parameters trainable; top blocks=[10, 11]
2026-10-19 16:32:30 - model - INFO - Instantiating Bidirectional LSTM model (CRF=False)

 student_distilled_bilstm_crf_False Parameters
  Total parameters:       20,353,044
  Trainable parameters:   20,353,044
  Frozen parameters:               0
  Trainable ratio:           100.00%

/usr/local/lib/python3.13/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _w

# 5. Evaluations

In [10]:
!python main.py \
  --mode evaluate \
  --model all

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
2026-10-19 16:58:42 - model - INFO - Instantiating LSTM model (CRF=False)
  -> Saved predictions to /kaggle/working/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora/results/predictions/lstm_predictions.txt
  -> Saved metrics to /kaggle/working/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora/results/metrics/lstm_token_metrics.csv
  -> Saved confusion matrix to /kaggle/working/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora/results/plots/lstm_confusion_matrix.png
  -> Saved entity distribution to /kaggle/working/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora/results/plots/lstm_entity_distribution.png
2026-10-19 16:58:49 - model - INFO - Instantiating Bidirectional LST

In [11]:
# knowledge distillation: Teacher - Phobert; Student - BiLSTM
!python main.py --mode evaluate --model bilstm \
  --checkpoint results/checkpoints/bilstm_distilled_best_model.pt

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
2026-10-19 17:03:10 - model - INFO - Instantiating Bidirectional LSTM model (CRF=False)
  -> Saved predictions to /kaggle/working/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora/results/predictions/bilstm_predictions.txt
  -> Saved metrics to /kaggle/working/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora/results/metrics/bilstm_token_metrics.csv
  -> Saved confusion matrix to /kaggle/working/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora/results/plots/bilstm_confusion_matrix.png
  -> Saved entity distribution to /kaggle/working/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora/results/plots/bilstm_entity_distribution.png


# 6. Inference example

In [12]:
!python main.py \
  --mode infer \
  --model all \
  --infer_text "Từ ngày 24-7 đến ngày 31-7, bệnh nhân được mẹ là bà H.T.P (47 tuổi) đón về nhà ở phường Phước Hoà (bằng xe máy), không đi đâu, chỉ ra tạp hoá Phượng, chợ Vườn Lài, phường An Sơn cùng mẹ bán tạp hoá ở đây."

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
2026-10-19 17:03:36 - model - INFO - Instantiating LSTM model (CRF=False)
Từ
ngày
24-7
đến
ngày
31-7 [B-DATE]
,
bệnh_nhân
được
mẹ
là
bà
H.T.P [B-NAME]
(
47 [B-AGE]
tuổi
)
đón
về
nhà
ở
phường [B-LOCATION]
Phước_Hòa [I-LOCATION]
(
bằng
xe_máy
)
,
không
đi
đâu
,
chỉ
ra
tạp
hóa_Phượng
,
chợ
Vườn_Lài [I-ORGANIZATION]
,
phường [B-LOCATION]
An_Sơn
cùng
mẹ
bán
tạp_hóa
ở
đây
.
2026-10-19 17:03:37 - model - INFO - Instantiating Bidirectional LSTM model (CRF=False)
Từ
ngày
24-7 [B-DATE]
đến
ngày
31-7 [B-DATE]
,
bệnh_nhân
được
mẹ
là
bà
H.T.P [B-NAME]
(
47 [B-AGE]
tuổi
)
đón
về
nhà
ở
phường [B-LOCATION]
Phước_Hòa [I-LOCATION]
(
bằng
xe_máy
)
,
không
đi
đâu
,
chỉ
ra
tạp
hóa_Phượng
,
chợ
Vườn_Lài [I-LOCATION]
,
phường
An_Sơn
cùng
mẹ
bán
tạp_hóa
ở


In [13]:
!python main.py --mode infer --model bilstm \
  --checkpoint results/checkpoints/bilstm_distilled_best_model.pt \
  --infer_text "Từ ngày 24-7 đến ngày 31-7, bệnh nhân được mẹ là bà H.T.P (47 tuổi) đón về nhà ở phường Phước Hoà (bằng xe máy), không đi đâu, chỉ ra tạp hoá Phượng, chợ Vườn Lài, phường An Sơn cùng mẹ bán tạp hoá ở đây."


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
2026-10-19 17:04:07 - model - INFO - Instantiating Bidirectional LSTM model (CRF=False)
Từ
ngày
24-7
đến
ngày
31-7 [B-DATE]
,
bệnh_nhân
được
mẹ
là
bà
H.T.P [B-NAME]
(
47 [B-AGE]
tuổi
)
đón
về
nhà
ở
phường [B-LOCATION]
Phước_Hòa [I-LOCATION]
(
bằng
xe_máy
)
,
không
đi
đâu
,
chỉ
ra
tạp
hóa_Phượng
,
chợ [B-LOCATION]
Vườn_Lài [I-LOCATION]
,
phường [B-LOCATION]
An_Sơn [I-LOCATION]
cùng
mẹ
bán
tạp_hóa
ở
đây
.


# 7. Compress results

In [14]:
!rm -rf .env results/checkpoints/ && zip -qr results.zip results/ && echo "Compressed successfully"

Compressed successfully
